# SAMATVA: free GPU render

This notebook renders every photoreal plate of the SAMATVA soul film on Kaggle's free graphics cards and packs them into one zip.

**Before you press Run all (one time only):**
1. Sign in to kaggle.com and verify your phone number (Settings → Phone verification). Without it, GPUs and internet stay switched off.
2. Create → New Notebook → File → Import Notebook, and choose this file.
3. In the right-hand panel, **Session options**:
   - set **Accelerator** to **GPU T4 x2**
   - set **Internet** to **On**
4. Press **Run all**.

**When it finishes,** open the **Output** panel (right side) and download `samatva_plates.zip`. Share that zip back (for example as a Google Drive link) and the film is assembled from it.

Expected time for the hero settings: about 30–60 minutes on two T4 cards. Kaggle gives about 30 free GPU hours a week.

## 1 · Settings
`hero` = full HD, 64 samples, native 24 fps (best). `film` = the faster settings used on the CPU machine.

In [ ]:
QUALITY = 'hero'            # 'hero' or 'film'
SHOTS   = ['reform', 'build', 'gen', 'loop', 'finale']
THERMAL = ['reform', 'build', 'gen']   # chapters that open in heat vision
DEVICE  = 'OPTIX'           # 'OPTIX' (fastest on T4) or 'CUDA'
REPO    = 'https://raw.githubusercontent.com/2025514153aashirwad-glitch/2025514153aashirwad-glitch.github.io/ccr-22a9ddef-0uao11'
HDRI_URL = 'https://dl.polyhaven.org/file/ph-assets/HDRIs/hdr/2k/brown_photostudio_02_2k.hdr'   # CC0, Poly Haven
BLENDER_URL = 'https://download.blender.org/release/Blender4.5/blender-4.5.16-linux-x64.tar.xz'
JPEG_QUALITY = 95           # plates are converted to JPEG to keep the zip small (about 1.5 GB for hero)

Q = {'hero': dict(res='1920x1080', fps=24, metal=64, thermal=16),
     'film': dict(res='1280x720',  fps=8,  metal=10, thermal=10)}[QUALITY]
LEN = {'reform': 14, 'build': 14, 'gen': 13, 'loop': 12, 'finale': 12}
TH_SECONDS = 5.5
print(QUALITY, Q)

## 2 · Check the graphics cards

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3 · Install Blender 4.5 LTS (about 2 minutes)

In [ ]:
import os, subprocess
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/blender/blender'):
    !wget -q {BLENDER_URL} -O /kaggle/blender.tar.xz
    !mkdir -p /kaggle/blender && tar xf /kaggle/blender.tar.xz -C /kaggle/blender --strip-components=1 && rm /kaggle/blender.tar.xz
!apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libxfixes3 libgl1 > /dev/null 2>&1 || true
!/kaggle/blender/blender -b --version | head -1

## 4 · Fetch the scene: traced Samatwam marks, loop path, shot script, studio light

In [ ]:
import urllib.request, pathlib
A = pathlib.Path('/kaggle/assets'); (A / 'vector').mkdir(parents=True, exist_ok=True)
files = [f'brand/logo/vector/{v}-{p}.svg' for v in ('sun', 'wheel', 'hexagon', 'octagon') for p in ('crescent', 'infinity', 'emblem', 'triangles')]
files += ['brand/logo/vector/sun-looppath.json']
for f in files:
    urllib.request.urlretrieve(f'{REPO}/{f}', A / 'vector' / pathlib.Path(f).name)
urllib.request.urlretrieve(f'{REPO}/films/samatva-soul/source/blender/soul_shots.py', A / 'soul_shots.py')
urllib.request.urlretrieve(HDRI_URL, A / 'studio.hdr')
print(len(list((A / 'vector').iterdir())), 'vector files ·', (A / 'studio.hdr').stat().st_size // 1_000_000, 'MB light map')

## 5 · Render (one queue per graphics card)
Each shot is split in half across the two cards. Re-running this cell resumes: finished frames are skipped.

In [ ]:
import subprocess, threading, time, math, os
OUT = pathlib.Path('/kaggle/working/plates'); OUT.mkdir(exist_ok=True)
env0 = dict(os.environ, SAMATVA_VEC=str(A / 'vector'), SAMATVA_HDRI=str(A / 'studio.hdr'), SAMATVA_DEVICE=DEVICE)
jobs = []
for s in THERMAL:
    n = math.ceil(TH_SECONDS * Q['fps']); jobs.append((s, 'thermal', n, Q['thermal']))
for s in SHOTS:
    jobs.append((s, 'metal', LEN[s] * Q['fps'], Q['metal']))
queues = [[], []]
for s, p, n, spp in jobs:            # split every job in two halves, one per card
    h = n // 2
    queues[0].append((s, p, 1, h, spp)); queues[1].append((s, p, h + 1, n, spp))
def worker(gpu, q):
    for s, p, a, b, spp in q:
        out = OUT / f'{s}-{p}'; out.mkdir(exist_ok=True); (out / 'fps.txt').write_text(str(Q['fps']))
        cmd = ['/kaggle/blender/blender', '-b', '--factory-startup', '-P', str(A / 'soul_shots.py'), '--', s, p, str(out),
               '--frames', f'{a}-{b}', '--samples', str(spp), '--res', Q['res'], '--fps', str(Q['fps'])]
        t0 = time.time(); r = subprocess.run(cmd, env=dict(env0, CUDA_VISIBLE_DEVICES=str(gpu)), capture_output=True, text=True)
        ok = 'DONE' in r.stdout
        print(f'GPU{gpu} {s:7s} {p:7s} {a:4d}-{b:4d}  {"ok" if ok else "FAILED"}  {(time.time() - t0) / 60:5.1f} min', flush=True)
        if not ok: print(r.stdout[-2000:], r.stderr[-2000:])
t0 = time.time()
th = [threading.Thread(target=worker, args=(g, queues[g])) for g in (0, 1)]
[t.start() for t in th]; [t.join() for t in th]
print(f'all plates rendered in {(time.time() - t0) / 60:.1f} min')

## 6 · Check: frame counts and a look at each shot

In [ ]:
from PIL import Image
from IPython.display import display
for d in sorted(OUT.iterdir()):
    fr = sorted(d.glob('*.png'))
    print(f'{d.name:16s} {len(fr):5d} frames')
    if fr:
        im = Image.open(fr[len(fr) // 2]); im.thumbnail((480, 270)); display(im)

## 7 · Pack the plates into one zip

In [ ]:
import shutil
PK = pathlib.Path('/kaggle/working/samatva_plates'); PK.mkdir(exist_ok=True)
for d in sorted(OUT.iterdir()):
    o = PK / d.name; o.mkdir(exist_ok=True); shutil.copy(d / 'fps.txt', o / 'fps.txt')
    for f in sorted(d.glob('*.png')):
        j = o / (f.stem + '.jpg')
        if not j.exists(): Image.open(f).convert('RGB').save(j, quality=JPEG_QUALITY, subsampling=0)
shutil.make_archive('/kaggle/working/samatva_plates', 'zip', PK)
shutil.rmtree(OUT); shutil.rmtree(PK)      # keep only the zip in Output
print(round(os.path.getsize('/kaggle/working/samatva_plates.zip') / 1e9, 2), 'GB → download it from the Output panel')